# EDA Cartier QTEM — Fase 8: Decision Map + Cleaning Pipeline

**Obiettivo**: documentare ogni decisione di cleaning e applicare il pipeline production-ready.  
**Principio**: legge da `data/raw/`, scrive in `data/processed/`. I file raw non vengono mai modificati.  
**Struttura**: 8A Decision Map → 8B-8C Cleaning → 8D Supplementary Features → 8E Validation → 8F Save.


In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from utils import OUT_TABLES
DATA_PROC = ROOT / 'data' / 'processed'
print(f"ROOT: {ROOT}")
print(f"Output tables: {OUT_TABLES}")
print(f"Data processed: {DATA_PROC}")

---
## 8A — Decision Map

La decision map trasforma ogni problema identificato nelle Fasi 1-7 in una decisione operativa con motivazione esplicita e impatto sul modello documentato.

**Principio**: la decision map viene scritta PRIMA del codice di cleaning — è la spec da cui deriva il codice.


In [ ]:
# Esegui il builder della decision map
import subprocess, sys
result = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'build_decision_map.py')],
                       capture_output=True, text=True, cwd=str(ROOT))
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

dm = pd.read_csv(OUT_TABLES / 'decision_map.csv')
print(f"\nDecision Map: {len(dm)} decisioni")
print(f"Dataset coinvolti: {dm['Dataset'].unique()}")
print(f"\nRiepilogo per tipo decisione:")
print(dm['Decisione'].str[:40].value_counts().to_string())

In [ ]:
# Visualizza decision map completa
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)
dm[['Dataset', 'Problema', 'N record', 'Decisione', 'Impatto modello']]

---
## 8B-8C — Cleaning Pipeline

Il cleaning è implementato in `scripts/cleaning.py` con funzioni indipendenti per ogni dataset.

Ogni funzione:
- Riceve il DataFrame raw come input
- Restituisce il DataFrame pulito (mai in-place)
- Stampa un report con N righe rimosse, flag create, trasformazioni applicate


In [ ]:
from cleaning import (
    clean_aggregated_data, clean_transactions, clean_clients,
    clean_crc, clean_ccp, build_supplementary_features,
    validate_cleaning, run_all_cleaning,
    DATA_RAW,
)

# Carica raw per avere riferimento di confronto
print("Caricamento dataset raw...")
raw = {}
raw['Aggregated_Data'] = pd.read_csv(DATA_RAW / 'Aggregated_Data.csv',
                                      parse_dates=['DATE_TARGET'], low_memory=False)
raw['Transactions']    = pd.read_csv(DATA_RAW / 'Transactions.csv',
                                      parse_dates=['TRS_DATE'], low_memory=False)
raw['Clients']         = pd.read_csv(DATA_RAW / 'Clients.csv',
                                      parse_dates=['BIRTH_DATE','FIRST_PURCHASE_DATE','FIRST_TRANSACTION_DATE'],
                                      low_memory=False)
raw['CRC']             = pd.read_csv(DATA_RAW / 'CRC.csv', parse_dates=['CREATION_DATE'], low_memory=False)
raw['CCP']             = pd.read_csv(DATA_RAW / 'CCP.csv', parse_dates=['CREATION_DATE','SALE_DATE'], low_memory=False)
print("Raw caricati.")

In [ ]:
# Applica cleaning per ogni dataset
clean = {}
clean['Aggregated_Data'] = clean_aggregated_data(raw['Aggregated_Data'])
clean['Transactions']    = clean_transactions(raw['Transactions'])
clean['Clients']         = clean_clients(raw['Clients'])
clean['CRC']             = clean_crc(raw['CRC'])
clean['CCP']             = clean_ccp(raw['CCP'])

In [ ]:
# Verifica flag binarie create
flag_report = []
flag_cols = {
    'Aggregated_Data': ['AGE_KNOWN', 'HAS_MULTIPLE_PURCHASES', 'HAS_REPAIR_HISTORY'],
    'Transactions':    ['TO_WITHOUTTAX_IMPUTED', 'SERIAL_NUMBER_KNOWN', 'WWPRICE_MISSING'],
    'Clients':         ['BIRTH_DATE_VALID', 'PURCHASE_DATE_ANOMALY'],
    'CRC':             ['HAS_APPOINTMENT_DURATION'],
    'CCP':             ['SALE_DATE_VALID'],
}
for ds, cols in flag_cols.items():
    for col in cols:
        if col in clean[ds].columns:
            vc = clean[ds][col].value_counts()
            pct_1 = vc.get(1, 0) / len(clean[ds]) * 100
            flag_report.append({'Dataset': ds, 'Flag': col,
                                 'N=0': vc.get(0, 0), 'N=1': vc.get(1, 0),
                                 '% flag=1': round(pct_1, 1)})

flag_df = pd.DataFrame(flag_report)
print("Riepilogo flag binarie create:")
print(flag_df.to_string(index=False))

---
## 8D — Supplementary Features

In [ ]:
supp = build_supplementary_features(clean['Clients'], clean['CRC'], clean['CCP'])
print(f"\nSupplementary features shape: {supp.shape}")
print(f"Colonne: {list(supp.columns)}")
print("\nStatistiche feature supplementari:")
for col in supp.columns:
    if col == 'CLIENT_ID':
        continue
    if supp[col].dtype in [np.int8, np.int64, int]:
        if supp[col].max() <= 1:
            print(f"  {col}: {supp[col].sum():,} con valore=1 ({supp[col].mean()*100:.1f}%)")
        else:
            print(f"  {col}: mean={supp[col].mean():.1f}, max={supp[col].max():,}")
    else:
        print(f"  {col}: mean={supp[col].mean():.1f}, null={supp[col].isna().sum():,}")
clean['supplementary_features'] = supp

---
## 8E — Validation Report

In [ ]:
val_df = validate_cleaning(raw, clean)
print(f"\nCheck totali: {len(val_df)} | PASS: {(val_df['Status']=='PASS').sum()} | FAIL: {(val_df['Status']=='FAIL').sum()}")
val_df.to_csv(OUT_TABLES / 'cleaning_validation_report.csv', index=False)

if (val_df['Status'] == 'FAIL').any():
    print("\nCHECK FALLITI:")
    print(val_df[val_df['Status'] == 'FAIL'].to_string(index=False))
else:
    print("\nTutti i check superati.")

---
## 8F — Salvataggio dataset processati

In [ ]:
DATA_PROC.mkdir(parents=True, exist_ok=True)

save_map = {
    'Aggregated_Data_clean': 'Aggregated_Data',
    'Transactions_clean':    'Transactions',
    'Clients_clean':         'Clients',
    'CRC_clean':             'CRC',
    'CCP_clean':             'CCP',
    'supplementary_features':'supplementary_features',
}

summary_rows = []
for fname, key in save_map.items():
    df_out = clean[key]
    path = DATA_PROC / f'{fname}.csv'
    df_out.to_csv(path, index=False)
    n_raw_r  = raw[key].shape[0] if key in raw else 0
    n_raw_c  = raw[key].shape[1] if key in raw else 0
    n_rem    = n_raw_r - df_out.shape[0]
    n_newcol = df_out.shape[1] - n_raw_c
    print(f"  {fname}.csv")
    print(f"    Shape: {df_out.shape[0]:,} righe x {df_out.shape[1]} colonne")
    print(f"    Righe rimosse: {n_rem:,} | Colonne aggiunte: {max(0, n_newcol)}")
    summary_rows.append({
        'File': f'{fname}.csv',
        'Righe': df_out.shape[0],
        'Colonne': df_out.shape[1],
        'Righe rimosse': n_rem,
        'Colonne aggiunte': max(0, n_newcol),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n=== RIEPILOGO SAVING ===")
print(summary_df.to_string(index=False))
print("\n=== FASE 8 COMPLETATA ===")